In [0]:
# %pip install transformers==4.49.0 torchvision==0.20.1 accelerate==1.4.0 evaluate datasets

In [0]:
# %pip install datasets==2.20.0 mlflow==2.20.2

In [0]:
# %restart_python

In [0]:
# ################### Read stream exaple
# training_df = spark.readStream.table("dbr_dev.joshuandegwa_silver.training_images")
# display(training_df.limit(1), 
#         checkpointLocation = '/Volumes/dbr_dev/joshuandegwa_bronze/images/_checkpoints/training_display')

In [0]:
# ==============================================================================
# PIPELINE: Spark Structured Stream Image Preprocessor & Resizer
# DESCRIPTION: Reads raw claim images from a Delta stream, applies a distributed 
# Pandas UDF to crop them to square dimensions, resizes them to 224x224 pixels, 
# and writes the cleaned JPEG bytes into the Silver layer Delta table.
# ==============================================================================

import io
from pyspark.sql.functions import pandas_udf, col
from PIL import Image

IMAGE_RESIZE = 224

# Your correct path structure
landing_catalog = "dbr_dev"
landing_schema = "joshuandegwa_bronze"
base_path = f"/Volumes/{landing_catalog}/{landing_schema}/images"
metadata_path = f"{base_path}/_checkpoints"

# 1. Define the source streaming DataFrame
training_df = spark.readStream.table("dbr_dev.joshuandegwa_silver.training_images")

# 2. Resize UDF function with safe error handling and bilinear interpolation
@pandas_udf("binary")
def resize_image_udf(content_series):
    def resize_image(content):
        if not content:
            return None
        try:
            # Load the PIL image
            image = Image.open(io.BytesIO(content))
            width, height = image.size   
            new_size = min(width, height)
            
            # Crop the center of the image
            image = image.crop(((width - new_size) / 2, (height - new_size) / 2, (width + new_size) / 2, (height + new_size) / 2))
            
            # Resize to target resolution using bilinear filtering for cleaner ML features
            image = image.resize((IMAGE_RESIZE, IMAGE_RESIZE), Image.Resampling.BILINEAR)
            
            # Save back as jpeg bytes
            output = io.BytesIO()
            image.save(output, format='JPEG')
            return output.getvalue()
        except Exception:
            # Safely skip/return None if an image is corrupted or truncated
            return None
            
    return content_series.apply(resize_image)

# Add metadata to enable native image preview in the Databricks UI
image_meta = {"spark.contentAnnotation" : '{"mimeType": "image/jpeg"}'}

# 3. Execute the stream write to your Silver layer target table
query = (
    training_df
      .withColumn("content", resize_image_udf(col("content")).alias("content", metadata=image_meta))
      .writeStream
      .format("delta")
      .outputMode("append")
      .option("checkpointLocation", f"{metadata_path}/training_resized_checkpoint")
      .trigger(availableNow=True)
      .toTable("dbr_dev.joshuandegwa_silver.training_images_resized")
)

# Wait for the micro-batch run to complete and exit cleanly
query.awaitTermination()

In [0]:
display(spark.table("dbr_dev.joshuandegwa_silver.training_images_resized").limit(10))

In [0]:
# ==============================================================================
# PIPELINE: Delta to Hugging Face Dataset Conversion & Experiment Setup
# DESCRIPTION: Sets up the MLflow tracking experiment, loads cleaned image bytes 
# from the Silver Delta table, converts them into a Hugging Face Dataset object, 
# and splits the data into training and validation sets.
# ==============================================================================

from datasets import Dataset
import mlflow

# Setup the training experiment under a standard shared or user path
mlflow.set_experiment("/Shared/image-claims-classifier")

# 1. Read the table and filter out any null/corrupted image rows
clean_df = spark.table("dbr_dev.joshuandegwa_silver.training_images_resized").filter("content is not null")

# 2. Convert to Hugging Face Dataset safely
dataset = Dataset.from_pandas(clean_df.toPandas()).rename_column("content", "image")

# 3. Create splits
splits = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = splits['train']
val_ds = splits['test']

In [0]:
# ==============================================================================
# PIPELINE: End-to-End Fine-Tuning, MLflow Packaging, and Unity Catalog Registration
# DESCRIPTION: Trains a fine-tuned ResNet-50 computer vision classifier, wraps 
# inference, logs artifacts with signatures, and registers the model directly 
# into Unity Catalog under the Gold layer with a "prod" alias.
# ==============================================================================

import io
import torch
import pandas as pd
import mlflow
from PIL import Image
from transformers import pipeline, Trainer
from mlflow.models import infer_signature
from mlflow import artifacts
from mlflow.tracking import MlflowClient

# 1. Define the custom MLflow wrapper class
class ModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, pipeline):
        self.pipeline = pipeline
        self.pipeline.model.eval()

    def predict(self, context, images):
        with torch.set_grad_enabled(False):
            # Convert raw byte streams to PIL images
            img_list = images['content'].apply(lambda b: Image.open(io.BytesIO(b)).convert("RGB")).to_list()
            predictions = self.pipeline(img_list)
            # Filter and return only the class with the highest confidence score
            return pd.DataFrame([max(r, key=lambda x: x['score']) for r in predictions])

# 2. Execute Training and MLflow Logging Run
with mlflow.start_run(run_name="hugging_face_claims_finetuned") as run:
    # Log input dataset lineage from Hugging Face
    mlflow.log_input(mlflow.data.from_huggingface(train_ds, "training"))

    # Custom collator to handle tensor stacking and one-hot encoding
    def collate_fn(examples):
        pixel_values = torch.stack([torch.tensor(e["pixel_values"]) for e in examples])
        labels = torch.tensor([e["labels"] for e in examples], dtype=torch.long)
        return {"pixel_values": pixel_values, "labels": labels}

    # Initialize Trainer using your clean image processor
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=image_processor,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,
    )
    
    # Train the model on the GPU cluster
    trainer.train()

    # Build the final HF pipeline using the best model checkpoint
    best_checkpoint = trainer.state.best_model_checkpoint or "./resnet-claims-finetuned"
    classifier = pipeline("image-classification", model=best_checkpoint, image_processor=image_processor)

    # Wrap the model and infer model signature using Silver Delta table data
    wrapped_model = ModelWrapper(classifier)
    test_df = spark.table("dbr_dev.joshuandegwa_silver.training_images_resized").select('content').toPandas()
    predictions = wrapped_model.predict(None, test_df)
    signature = infer_signature(test_df, predictions)

    # Extract standard pip requirements
    reqs = mlflow.transformers.get_default_pip_requirements(model)

    # Log the packaged model to MLflow
    logged = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=wrapped_model,
        pip_requirements=reqs,
        signature=signature,
    )

# 3. Sanity-check output URIs and artifacts
print("logged.model_uri:", logged.model_uri)
print("logged.run_id  :", logged.run_id)
print("model files    :", artifacts.list_artifacts(logged.model_uri))

# 4. Register Model to Unity Catalog & Assign 'prod' Alias
mlflow.set_registry_uri("databricks-uc")
model_name = "dbr_dev.joshuandegwa_gold.claims_damage_level"

registered = mlflow.register_model(
    model_uri=logged.model_uri,
    name=model_name,
)

MlflowClient().set_registered_model_alias(
    name=model_name,
    alias="prod",
    version=registered.version,
)

print(f"Successfully registered {model_name} v{registered.version} to Unity Catalog and set alias 'prod'.")

In [0]:
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import StructType, StructField, StringType, FloatType
import pandas as pd

DAMAGE_PRED_SCHEMA = StructType([
    StructField("label", StringType()),
    StructField("score", FloatType()),
])

# 1. Define a pandas UDF that blocks the ABI-incompatible flash_attn before
#    loading the model — this fix must run inside the worker process
@pandas_udf(DAMAGE_PRED_SCHEMA)
def predict_damage_udf(content: pd.Series) -> pd.DataFrame:
    import sys
    sys.modules.setdefault("flash_attn", None)
    sys.modules.setdefault("flash_attn_2_cuda", None)
    import mlflow
    model = mlflow.pyfunc.load_model("models:/dbr_dev.joshuandegwa_gold.claims_damage_level@prod")
    return model.predict(pd.DataFrame({"content": content}))

# 2. Run distributed batch inference over the Silver layer and save to the Gold predictions table
(
    spark.table("dbr_dev.joshuandegwa_silver.training_images_resized")
    .withColumn("damage_prediction", predict_damage_udf(col("content")))
    .write
    .mode("overwrite")
    .saveAsTable("dbr_dev.joshuandegwa_gold.damage_predictions")
)

print("Batch inference completed successfully and saved to dbr_dev.joshuandegwa_gold.damage_predictions!")

In [0]:
# Load and display the Gold predictions table
predictions = spark.table('dbr_dev.joshuandegwa_gold.damage_predictions')
display(predictions)

In [0]:
# 1. Load the Gold predictions table and flatten the damage_prediction struct
predictions = spark.table('dbr_dev.joshuandegwa_gold.damage_predictions')

results = (
    predictions
    .selectExpr("path", "label", "damage_prediction.label as predictions", "damage_prediction.score as score")
    .toPandas()
)

# 2. Display the resulting evaluation dataframe
display(results)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create confusion matrix
conf_matrix = pd.crosstab(results['label'], results['predictions'])

# Plot confusion matrix with labels and clean sizing
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, cmap="Blues", fmt='d', cbar=False)

plt.title("Claims Damage Classification — Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import StructType, StructField, StringType, FloatType
import pandas as pd

DAMAGE_PRED_SCHEMA = StructType([
    StructField("label", StringType()),
    StructField("score", FloatType()),
])

# 1. Define a pandas UDF that blocks the ABI-incompatible flash_attn before
#    loading the model — this fix must run inside the worker process
@pandas_udf(DAMAGE_PRED_SCHEMA)
def predict_damage_udf(content: pd.Series) -> pd.DataFrame:
    import sys
    sys.modules.setdefault("flash_attn", None)
    sys.modules.setdefault("flash_attn_2_cuda", None)
    import mlflow
    model = mlflow.pyfunc.load_model("models:/dbr_dev.joshuandegwa_gold.claims_damage_level@prod")
    return model.predict(pd.DataFrame({"content": content}))

# 3. Read raw claim images from the Silver layer and apply the model UDF to generate predictions
raw_images_df = (
    spark.table("dbr_dev.joshuandegwa_silver.claim_images")
    .withColumn("damage_prediction", predict_damage_udf(col("content")))
)

# 4. Load auxiliary claim metadata from the Bronze layer
metadata_df = spark.table("dbr_dev.joshuandegwa_bronze.claim_metadata")

# 5. Join the predicted images with metadata on the shared 'image_name' key 
#    and save the final enriched dataset to the Gold layer
(
    raw_images_df
    .join(metadata_df, on="image_name", how="inner")
    .write
    .mode('overwrite')
    .saveAsTable("dbr_dev.joshuandegwa_gold.claim_images_predicted")
)

print("Successfully generated predictions, joined with metadata, and saved to dbr_dev.joshuandegwa_gold.claim_images_predicted!")

In [0]:
# Load a sample from the final Gold predictions and metadata table
final_preview_df = spark.table("dbr_dev.joshuandegwa_gold.claim_images_predicted")

# Display the first few rows
display(final_preview_df.limit(10))

In [0]:
%sql
select * from dbr_dev.joshuandegwa_bronze.claim_metadata limit 100;

In [0]:
# 1. Check how many rows are in your Silver images table
print("Silver image rows:", spark.table("dbr_dev.joshuandegwa_silver.claim_images").count())

# 2. Check how many rows are in your Bronze metadata table
# print("Bronze metadata rows:", spark.table("dbr_dev.joshuandegwa_bronze.claim_metadata").count())

# 3. Preview a few sample values of the join key from both sides to check formatting
print("--- Silver Image Names ---")
spark.table("dbr_dev.joshuandegwa_silver.claim_images").select("image_name").show(5, truncate=False)

print("--- Bronze Metadata Image Names ---")
# spark.table("dbr_dev.joshuandegwa_bronze.claim_images_meta").select("image_name").show(5, truncate=False)